In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    # f.read()가 파일 내용을 담은 문자열 객체를 새로 만들고, raw_text라는 변수가 그 객체를 가리킨다
    raw_text = f.read()

print("총 문자 개수: ", len(raw_text))
print(raw_text[:99])

총 문자 개수:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
import re

text = "Hello, world. This, is a test"

# re.split(패턴, 문자열): 문자열을 '패턴'을 기준으로 잘라서 리스트로 반환
#
# r'(\s)'  ← 이 부분이 정규표현식 패턴
#   ┌─ r'...'  : raw 문자열(raw string). 역슬래시(\)를 특수 처리 없이 그대로 전달한다.
#   │            정규표현식엔 \ 가 자주 쓰여서, 항상 r'...' 형태로 감싸는 게 관례다.
#   │
#   ├─ \s     : 공백 문자(whitespace) 하나를 의미하는 메타문자.
#   │            스페이스( ), 탭(\t), 줄바꿈(\n) 등이 모두 여기 해당한다.
#   │            (참고: \S 는 대문자라 반대로 '공백이 아닌 문자'를 뜻한다.)
#   │
#   └─ ( )    : 캡처 그룹(capture group). 괄호로 감싼 부분을 '기억'한다.
#                re.split에서 패턴을 괄호로 감싸면, 잘라내는 기준이 된
#                구분자(여기선 공백) 자체도 결과 리스트에 함께 포함된다.
#                → 괄호가 없으면 공백은 버려지고, 있으면 공백도 살아남는다.
result = re.split(r"(\s)", text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test']


In [4]:
# re.split(패턴, 문자열): '패턴'에 걸리는 부분을 기준으로 문자열을 잘라 리스트로 반환
#
# r"[,.]|\s"  ← 정규표현식 패턴
#   ┌─ r"..."  : raw 문자열. 역슬래시(\)를 있는 그대로 전달한다(정규식의 관례).
#   │
#   ├─ [,.]    : 문자 클래스(character class). 대괄호 안의 문자 중 '아무거나 하나'와 매칭.
#   │             여기서는 쉼표( , ) 또는 마침표( . ) 하나를 의미한다.
#   │             ※ 대괄호 [ ] 안에서는 . 이 특수문자가 아니라 '진짜 마침표'로 취급된다.
#   │               (밖에서 . 은 '아무 문자나'라는 뜻이지만, [ ] 안에선 문자 그대로다.)
#   │
#   ├─ |       : OR(또는). 왼쪽 패턴이나 오른쪽 패턴 중 하나에 걸리면 매칭.
#   │
#   └─ \s      : 공백 문자 하나(스페이스, 탭, 줄바꿈 등).
#
#   종합하면: "쉼표 또는 마침표 또는 공백" 중 하나를 만나면 그 자리를 기준으로 자른다.
#
result = re.split(r"([,.]|\s)", text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test']


In [5]:
# 리스트 컴프리헨션: result의 각 item을 순회하며 조건을 통과한 것만 새 리스트로 모은다
#
# item             → 원본 조각을 그대로 유지 (strip한 값이 아니라 원본을 담는다)
# if item.strip()  → 필터 조건. item의 앞뒤 공백을 제거해봤을 때 '내용이 남으면' True
#
#   ┌─ 'Hello' .strip() → 'Hello'  → 비어있지 않음 → True  → 살림
#   ├─ ','     .strip() → ','      → 비어있지 않음 → True  → 살림 (구두점은 유지!)
#   ├─ ''      .strip() → ''       → 빈 문자열     → False → 버림
#   └─ ' '     .strip() → ''       → 공백뿐이라 빈값 → False → 버림
#
#   ※ 빈 문자열 ''과 공백뿐인 ' '은 파이썬에서 거짓(False)으로 취급되어 걸러진다.
#   ※ 여기서 핵심: item.strip()은 '판단용'으로만 쓰고, 실제로 담기는 값은 원본 item이다.
#     그래서 쉼표·마침표는 그대로 살아남고, 의미 없는 빈칸·공백만 제거된다.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test']


In [6]:
text = "Hello, world. Is this-- a test?"

# re.split(패턴, 문자열): '패턴'에 걸리는 부분을 기준으로 문자열을 잘라 리스트로 반환
#
# r'([,.:;?_!"()\']|--|\s)'  ← 정규표현식 패턴
#
#   전체가 괄호( )로 감싸여 있음 → 캡처 그룹 → 잘라내는 기준(구분자)도 결과에 남는다.
#
#   내부는 |(OR)로 세 덩어리가 연결되어 있다:  A | B | C
#   → "A 또는 B 또는 C 중 하나에 걸리면 그 자리를 기준으로 자른다"
#
#   ┌─ A: [,.:;?_!"()\']  ← 문자 클래스. 대괄호 안 문자 중 '아무거나 하나'와 매칭.
#   │       하나씩 뜯어보면 아래 구두점들을 의미한다:
#   │         ,  쉼표        .  마침표      :  콜론       ;  세미콜론
#   │         ?  물음표      _  밑줄        !  느낌표
#   │         "  큰따옴표    (  여는 괄호   )  닫는 괄호
#   │         \' 작은따옴표(어퍼스트로피)
#   │           └─ \' 로 쓴 이유: 패턴 전체를 r'...' (작은따옴표)로 감쌌기 때문에,
#   │              안에서 작은따옴표를 문자로 쓰려면 \ 로 이스케이프해야 문자열이
#   │              중간에 끊기지 않는다. (정규식 규칙이라기보다 파이썬 문자열 규칙)
#   │           ※ 대괄호 [ ] 안에서는 . ? ( ) 같은 특수문자도 '문자 그대로'로 취급되어
#   │             일일이 이스케이프할 필요가 없다.
#   │
#   ├─ B: --  ← 하이픈 두 개(대시). '--' 라는 두 글자 연속을 하나의 구분자로 취급.
#   │           (한 개짜리 - 가 아니라 정확히 '--' 일 때만 매칭)
#   │
#   └─ C: \s  ← 공백 문자 하나(스페이스, 탭, 줄바꿈 등).
#
#   종합: 구두점 각각, 또는 '--', 또는 공백을 만나면 그 자리를 기준으로 자르고,
#         (캡처 그룹이므로) 그 구분자들도 결과 리스트에 함께 남긴다.
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)

# 앞뒤 공백을 벗겨(strip) 판단했을 때 내용이 남는 조각만 유지
#   → 공백뿐인 ' ' 이나 빈 문자열 '' 은 걸러지고, 단어·구두점·'--' 만 살아남는다
result = [item.strip() for item in result if item.strip()]
# result = [item for item in result if item.strip()]

print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [8]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

# 'Hello' → ③ 'Hello'.strip()='Hello' → 내용 있음 → True  → ① 'Hello'.strip() → 'Hello' 담김
# ''      → ③ ''.strip()=''           → 빈값     → False → 버려짐 (①까지 안 감)
# ' '     → ③ ' '.strip()=''          → 공백벗기니 빈값 → False → 버려짐
# 'world' → ③ 'world'.strip()='world' → 내용 있음 → True  → ① 담김
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed)) # 4690

print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
